# Custom Model Extension

Audience:
- Users who want to validate a raw PyTorch `nn.Module` inside the scDLKit workflow without registering it as a built-in model.

Prerequisites:
- Install `scdlkit[tutorials]`.
- Understand the PBMC quickstart first.
- Be comfortable reading small PyTorch modules.

Learning goals:
- load PBMC data with Scanpy
- define a small custom autoencoder in pure PyTorch
- wrap it with `wrap_reconstruction_module(...)`
- train it for the `representation` task through `Trainer`
- push the learned latent space back into `adata.obsm`
- continue with Scanpy neighbors and UMAP

Out of scope:
- plugin packaging
- full custom training loops outside scDLKit
- proving the custom model is better than all built-in baselines

This is the supported custom-model path in `v0.1.3`. `TaskRunner` remains the built-in beginner API; custom wrapped modules are supported through `Trainer` first.

Related APIs:
- `Trainer`: lower-level stable training loop
- `wrap_reconstruction_module(...)`: adapter path for raw PyTorch modules

Next steps:
- Tutorial: `classification_demo.ipynb` or `scgpt_cell_type_annotation.ipynb`
- API: `docs/api/adapters.md`


## Install

```bash
python -m pip install "scdlkit[tutorials]"
```

This notebook is CPU-friendly. If a CUDA build of PyTorch is installed, the same code path uses GPU automatically through `device="auto"`.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import scanpy as sc
import torch
from torch import nn

from scdlkit import Trainer, prepare_data
from scdlkit.adapters import wrap_reconstruction_module
from scdlkit.data import transform_adata
from scdlkit.evaluation import evaluate_predictions, save_markdown_report, save_metrics_table
from scdlkit.visualization import plot_losses

OUTPUT_DIR = Path("artifacts/custom_model_extension")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_name}")


## Load PBMC data

We reuse `scanpy.datasets.pbmc3k_processed()` so the downstream analysis flow stays familiar.


In [ ]:
adata = sc.datasets.pbmc3k_processed()
adata


## Prepare the splits

For the adapter path we use the lower-level API directly: `prepare_data(...)`, then `Trainer(...)`.


In [ ]:
prepared = prepare_data(adata, label_key="louvain")
prepared.input_dim


## Define a custom PyTorch model

This is a plain `nn.Module`, not a built-in scDLKit model class.


In [ ]:
class CustomAutoencoder(nn.Module):
    def __init__(self, input_dim: int, latent_dim: int = 16) -> None:
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim),
        )

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        latent = self.encode(x)
        return self.decoder(latent)


module = CustomAutoencoder(prepared.input_dim, latent_dim=16)
module


## Wrap the module and train it

The wrapper gives the raw module the output and loss contract expected by scDLKit.


In [ ]:
model = wrap_reconstruction_module(
    module,
    input_dim=prepared.input_dim,
    supported_tasks=("representation", "reconstruction"),
)

trainer = Trainer(
    model=model,
    task="representation",
    epochs=10,
    batch_size=128,
    device="auto",
    seed=42,
)
trainer.fit(prepared.train, prepared.val)
trainer.history_frame.tail()


## Save a loss curve


In [ ]:
loss_fig, _ = plot_losses(trainer.history_frame)
loss_fig.savefig(OUTPUT_DIR / "loss_curve.png", dpi=150, bbox_inches="tight")
plt.close(loss_fig)
OUTPUT_DIR / "loss_curve.png"


## Evaluate the held-out split


In [ ]:
split = prepared.test or prepared.val or prepared.train
predictions = trainer.predict_dataset(split)
metrics = evaluate_predictions("representation", predictions)
metrics


## Push the latent space back into Scanpy

This is the important handoff: scDLKit trains the model, then Scanpy continues the downstream embedding workflow from `adata.obsm`.


In [ ]:
full_split = transform_adata(
    adata,
    prepared.preprocessing,
    label_encoder=prepared.label_encoder,
    batch_encoder=prepared.batch_encoder,
)
full_predictions = trainer.predict_dataset(full_split)
adata.obsm["X_scdlkit_custom"] = full_predictions["latent"]

sc.pp.neighbors(adata, use_rep="X_scdlkit_custom")
sc.tl.umap(adata, random_state=42)

umap_fig = sc.pl.umap(adata, color="louvain", return_fig=True, frameon=False)
umap_fig.savefig(OUTPUT_DIR / "latent_umap.png", dpi=150, bbox_inches="tight")
plt.close(umap_fig)
OUTPUT_DIR / "latent_umap.png"


## Save a simple report

This uses the public low-level evaluation helpers rather than `TaskRunner`.


In [ ]:
report_md = OUTPUT_DIR / "report.md"
report_csv = OUTPUT_DIR / "report.csv"

save_markdown_report(
    metrics,
    path=report_md,
    title="scDLKit custom model extension report",
    extra_sections=[
        "## Notes",
        "",
        "- Wrapped model type: `CustomAutoencoder`",
        "- Training surface: `Trainer`",
        "- Task: `representation`",
    ],
)
save_metrics_table(metrics, report_csv)

report_md, report_csv


## Expected outputs

This tutorial should leave four artifacts under `artifacts/custom_model_extension/`:

- `report.md`
- `report.csv`
- `loss_curve.png`
- `latent_umap.png`

Validation checklist:

1. The loss curve should decrease.
2. The latent UMAP should show broad structure rather than collapse.
3. The saved report should make it easy to compare your wrapped model against the built-in baselines.

Recommended next tutorials:
- PBMC model comparison to benchmark against built-in baselines
- reconstruction sanity check if your custom model also exposes reconstructed expression


In [ ]:
output_paths = {
    "report_md": str(report_md),
    "report_csv": str(report_csv),
    "loss_curve_png": str(OUTPUT_DIR / "loss_curve.png"),
    "latent_umap_png": str(OUTPUT_DIR / "latent_umap.png"),
}
output_paths
